# S2 — baseline audit of the clean baseline (seed 42) before the LOCK. No diffusion training. Test set NOT used.
Validation = 15 % of the clean TRAIN patients (fixed seed), used only for classifiers, gates and the guidance choice.

| test | what | rule (pre-stated in the script) |
|---|---|---|
| A10 | real data by class: view, laterality, density, digitiser (CPU) | reported with patient-bootstrap CI |
| A6 | pathology classifier (ConvNeXt-Tiny, fixed recipe, 3 seeds) | judge usable if mean **validation** AUC ≥ 0.65 |
| A6b | view classifier CC vs MLO | usable if validation AUC ≥ 0.95 |
| A7 | samples at guidance 4, 1 (200/class), 2, 3 (100/class); guidance 0 same-noise **identity test** | guidance 0 must give identical benign/malignant images |
| A9 | conditioning gate, view share, brightness/saturation, metric sanity, calibration KID vs train, memorisation, guidance rule | gate: AUC_gen − 0.5 ≥ 0.5·(AUC_val − 0.5) |
| A8 | Fokker–Planck residual, step-table spot check, S_ref(t), FP-vs-base gradients → λ candidates | spot check ≤ 0.10 |

**How to run**
1. Settings → Accelerator **GPU T4 ×2** · Internet **ON**.
2. Add Input → *Your Work* → **clean-data-baseline-retraining** (latest version) and Add Input → *CBIS-DDSM Breast Cancer Image Dataset*.
3. **Save Version → Save & Run All (Commit)**. Then **stop the interactive session** (it also uses GPU quota).

**Time:** expected ≈ 3 h; hard limit `BUDGET_H = 6.0` h in Cell 1 (every job is skipped if it would pass the limit).
**Send back:** `S2/S2_report.txt` + the `S2/grid_*.png` and `S2/memorisation_top8_*.png` images.

In [ ]:
# ===================== CELL 1 — CONFIG =====================
BUDGET_H = 6.0            # hard limit for this session (your quota showed 7 h left -> 7 - 0.8 margin, rounded down)
import os, time
os.environ["PA_OUT"] = "/kaggle/working/S2"
os.makedirs(os.environ["PA_OUT"], exist_ok=True)
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
def used_h():
    return (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
print("S2 baseline audit | budget", BUDGET_H, "h | output", os.environ["PA_OUT"])

In [ ]:
# ===================== CELL 2 — setup (needs Internet ON) =====================
import os, re, sys, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("/kaggle/working/U24_synthesis/.git"):
    subprocess.run("git clone -q https://github.com/Its-Me-Sk/U24_synthesis.git", shell=True, check=True)
if not os.path.exists("/tmp/pip_done"):
    # Kaggle runs Python 3.13: install the repo requirements once, never force-reinstall numpy / pandas
    subprocess.run("cd /kaggle/working/U24_synthesis && grep -v xformers requirements.txt > /tmp/req.txt && "
                   "pip install -q -r /tmp/req.txt", shell=True, check=True)
    subprocess.run("pip install -q --no-deps torch-fidelity", shell=True, check=True)
    open("/tmp/pip_done", "w").write("ok")
probe = ("import sys, numpy, pandas, scipy, torch, torchmetrics, monai, generative, PIL; "
         "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
         "'| torch', torch.__version__, '| torchmetrics', torchmetrics.__version__, '| monai', monai.__version__)")
env_chk = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
if env_chk.returncode != 0:
    print(env_chk.stderr[-2500:])
    raise SystemExit("!! environment check failed (see the error above) -> send Claude this block")
print("environment OK:", env_chk.stdout.strip())
p = "/kaggle/working/U24_synthesis/lightning_synthesis/model_architectures/monai_ddpm.py"
s = open(p).read()
s2 = re.sub(r"use_flash_attention\s*=\s*True", "use_flash_attention=False", s)   # xformers is not installed
if s2 != s:
    open(p, "w").write(s2)
print("flash attention ->", "patched off" if s2 != s else "already off")
m = "/kaggle/working/U24_synthesis/models.py"          # oracle loader (used later in A6): hard-coded config path
s = open(m).read()
s2 = re.sub(r"open\(\s*['\"]/home/[^'\"]*config\.yaml['\"]", "open('/kaggle/working/U24_synthesis/config.yaml'", s)
if s2 != s:
    open(m, "w").write(s2)
print("Cell 2 done. Kernel restart NOT needed (all GPU work runs in subprocesses).")

In [ ]:
# ===================== CELL 3 — code + clean data (reused) + input checks + job queues =====================
import os, sys, glob, json, shutil, subprocess, threading, time
from pathlib import Path
WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
os.chdir(WD)
FILES = {"phaseA_common.py": r'''# phaseA_common.py - shared helpers for the Phase A diagnostics (no training happens in Phase A).
# Model, schedule, data domain and conditioning are taken from the Experiment-2 code (train_arm2.py / eval_arm.py):
#   * model: model_architectures.MonaiDDPM().unet (MONAI-Generative DiffusionModelUNet, flash attention off)
#   * schedule: MonaiDDPM().scheduler.alphas_cumprod (linear beta 1e-4..0.02, T=1000)
#   * images: per-image min-max to [-1,1]; physics statistic on (x+1)/2
#   * conditioning: context = label as float (B,1,1); -1 = null label (classifier-free guidance)
import os, sys, glob, json, math, time
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
if WD not in sys.path:
    sys.path.insert(0, WD)
OUT = os.environ.get("PA_OUT", "/kaggle/working/phaseA")
os.makedirs(OUT, exist_ok=True)


def log(tag, *m):
    print(f"[{tag}]", *m, flush=True)


_INDEX = None


def find(name, roots=("/kaggle/input", "/kaggle/working")):
    """all files with this exact name in the attached inputs (one directory walk per process; image folders skipped)"""
    global _INDEX
    if _INDEX is None:
        _INDEX = {}
        for r in roots:
            for dp, dns, fns in os.walk(r):
                dns[:] = [d for d in dns if d not in ("jpeg", "U24_synthesis", ".git", "phaseA")]
                for fn in fns:
                    _INDEX.setdefault(fn, []).append(os.path.join(dp, fn))
    return sorted(set(_INDEX.get(name, [])))


def main_ckpt():
    """full-data Experiment-2 baseline, epoch 40 (the main Phase A checkpoint)"""
    h = find("base2_w_ep040.pt")
    if not h:
        raise SystemExit("base2_w_ep040.pt not found: Add Input -> Your Work -> the Experiment 2 notebook "
                         "(notebook64a7f79592), then run again")
    return h[0]


def clean_ckpt(seed, epoch=40):
    """clean-data baseline (step 2 of the timeline): base_clean_s<seed>_w_ep<epoch>.pt"""
    h = find(f"base_clean_s{seed}_w_ep{epoch:03d}.pt")
    if not h:
        raise SystemExit(f"base_clean_s{seed}_w_ep{epoch:03d}.pt not found: Add Input -> Your Work -> the clean-baseline "
                         "notebook (its latest output version), then run again")
    return h[0]


def optional_25_ckpt():
    """any 25 %-data baseline at epoch 45 from Experiment 3 or 4 (optional)"""
    for s in (42, 43, 44, 45, 46, 47, 48):
        h = find(f"base3_s{s}_w_ep045.pt")
        if h:
            return h[0]
    return None


def schedule(dev):
    from model_architectures import MonaiDDPM
    m = MonaiDDPM(lr=1e-5, T=1000)
    ac = m.scheduler.alphas_cumprod.detach().double().to(dev)
    unet = m.unet
    del m
    return unet, ac


def load_model(path, dev, dtype=torch.float32):
    """returns (unet in eval mode, alphas_cumprod float64, snapshot meta)"""
    snap = torch.load(path, map_location="cpu", weights_only=False)
    unet, ac = schedule(dev)
    sd = snap["unet"] if "unet" in snap else snap["models"]["net"]
    unet.load_state_dict({k: v.float() for k, v in sd.items()})
    unet = unet.to(dev).to(dtype).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    meta = {k: snap[k] for k in ("epoch", "step", "lam", "tau") if k in snap}
    meta["nlf"] = snap.get("nlf")
    meta["path"] = path
    del snap
    return unet, ac, meta


def load_cache(split):
    d = np.load(f"/tmp/{split}_cache.npz")
    return d["x"], d["y"]


def load_clean_meta(split):
    """clean cache WITH patient ids and md5 - same order as /tmp/<split>_cache.npz (written by prepare_clean.py)"""
    d = np.load(f"/kaggle/working/clean_data/{split}_cache.npz")
    return d["x"], d["y"], d["patient"].astype(str), d["md5"].astype(str)


def val_mask(patients, frac=0.15, seed=0):
    """patient-level VALIDATION split of the clean TRAIN set (classifiers, gates, guidance choice only).
    The diffusion model itself was trained on all of train. Fixed seed -> identical in every script/session."""
    ups = sorted(set(patients))
    rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])


def minmax_pm1(x):
    """x: torch (B,1,H,W) any range -> per-image min-max to [-1,1] (training domain)"""
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1


def to_model_domain(x_u8, dev):
    return minmax_pm1(torch.from_numpy(x_u8).float().unsqueeze(1).to(dev))


def ctx_of(labels, dev, dtype=torch.float32):
    """labels: list/array of 0, 1 or -1 (null) -> context tensor (B,1,1)"""
    return torch.as_tensor(np.asarray(labels, dtype=np.float32), device=dev).view(-1, 1, 1).to(dtype)


def eps_pred(unet, x, t_int, ctx, amp=False, chunk=8):
    """epsilon prediction. amp=True: exactly as in training (fp32 weights + fp16 autocast); amp=False: the weights'
    own dtype. Large batches are split into chunks (the non-flash attention at 64x64 needs ~0.5 GB per image)."""
    if chunk and x.shape[0] > chunk:
        if ctx.shape[0] == 1:
            ctx = ctx.expand(x.shape[0], *ctx.shape[1:])
        return torch.cat([eps_pred(unet, x[i:i + chunk], t_int, ctx[i:i + chunk], amp, None)
                          for i in range(0, x.shape[0], chunk)])
    tb = torch.full((x.shape[0],), int(t_int), device=x.device, dtype=torch.long)
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            out = unet(x, timesteps=tb, context=ctx)
        return out.float()
    return unet(x, timesteps=tb, context=ctx)


def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(obj, fh, indent=1, default=float)
''',
         "fp_core.py": r'''# fp_core.py - the Fokker-Planck (score-FPE) residual and its exact "manual recompute" gradient.
# Shared by the Phase A-2 diagnostics and (later) the Phase B/C training, so both use byte-identical maths.
# VP-SDE of the DDPM (tau = t/T, beta(tau) = -d log abar / d tau), score s = -eps_hat / sqrt(1 - abar_t):
#     d_tau s = 1/2 beta grad_x( x.s + div s + |s|^2 )
# projected on a unit direction u, Hutchinson probe v (Rademacher) for d_u(div s):
#     r(u,v) = u.d_tau s - 1/2 beta [ u.s + (J_s u).(x + 2 s) + v.(d_u J_s) v ]
# all derivatives by central finite differences (per-pixel step p; h_u = 256 p for unit u on 256x256, h_v = p);
# d_tau s by +-k integer steps. Product of two independent probes = unbiased estimate of (u.R)^2.
# Verified in Phase A: unit test vs autograd (1e-13), analytic Gaussian score -> residual ~0, wrong score -> O(1).
import math
import torch

T_STEPS = 1000
DTAU = 1.0 / T_STEPS


def p_of_t(t):
    """finite-difference step per t (fixed before Phase B from the A4 accuracy table: <= 1 % error vs the exact JVP)"""
    return 1e-3 if t <= 20 else (3e-3 if t <= 50 else 1e-2)


def beta_eff(logab, t):
    return float(-(logab[t + 1] - logab[t - 1]) / (2 * DTAU))


def plan(x, t, u, v1, v2, p, k=1):
    hu, hv = 256.0 * p, p
    E = [("tp", x, t + k), ("tm", x, t - k), ("s0", x, t), ("up", x + hu * u, t), ("um", x - hu * u, t)]
    for j, v in ((1, v1), (2, v2)):
        E += [(f"pp{j}", x + hu * u + hv * v, t), (f"pm{j}", x + hu * u - hv * v, t),
              (f"mp{j}", x - hu * u + hv * v, t), (f"mm{j}", x - hu * u - hv * v, t)]
    return E


def residuals(s, x, u, v1, v2, p, k, beta):
    hu, hv = 256.0 * p, p
    A = (u * (s["tp"] - s["tm"])).sum() / (2 * k * DTAU)
    Ju = (s["up"] - s["um"]) / (2 * hu)
    base = (u * s["s0"]).sum() + (Ju * (x + 2 * s["s0"])).sum()
    out = []
    for j, v in ((1, v1), (2, v2)):
        D2 = (v * (s[f"pp{j}"] - s[f"pm{j}"] - s[f"mp{j}"] + s[f"mm{j}"])).sum() / (4 * hu * hv)
        out.append(A - 0.5 * beta * (base + D2))
    return out[0], out[1], A


def coeffs(s, x, u, v1, v2, p, k, beta, r1, r2, w=1.0):
    """dL/ds_k for L = w r1 r2 (shared evaluations: w (r1 + r2) dr/ds_k; probe-specific: w r_other dr_j/ds_k)"""
    hu, hv = 256.0 * p, p
    b = 0.5 * beta
    Ju = (s["up"] - s["um"]) / (2 * hu)
    shared = {"tp": u / (2 * k * DTAU), "tm": -u / (2 * k * DTAU), "s0": -b * (u + 2 * Ju),
              "up": -b * (x + 2 * s["s0"]) / (2 * hu), "um": b * (x + 2 * s["s0"]) / (2 * hu)}
    g = {key: w * (r1 + r2) * val for key, val in shared.items()}
    for j, v, other in ((1, v1, r2), (2, v2, r1)):
        c = b * v / (4 * hu * hv)
        g[f"pp{j}"], g[f"pm{j}"], g[f"mp{j}"], g[f"mm{j}"] = -w * other * c, w * other * c, w * other * c, -w * other * c
    return g


def evaluate_nograd(sfn, E):
    s = {}
    with torch.no_grad():
        for tt in sorted({e[2] for e in E}):
            keys = [e[0] for e in E if e[2] == tt]
            out = sfn(torch.cat([e[1] for e in E if e[2] == tt]), tt)
            for i, kk in enumerate(keys):
                s[kk] = out[i:i + 1]
    return s


def fp_loss_and_grad(sfn, x, t, u, v1, v2, p, k, beta, w=1.0, group=1, backward=True):
    """exact gradient of L = w r1 r2 accumulated into .grad (memory of ONE network evaluation). Returns (L, r1, r2, lhs)."""
    E = plan(x, t, u, v1, v2, p, k)
    s = evaluate_nograd(sfn, E)
    with torch.no_grad():
        r1, r2, A = residuals(s, x, u, v1, v2, p, k, beta)
        g = coeffs(s, x, u, v1, v2, p, k, beta, r1, r2, w)
    if backward:
        for tt in sorted({e[2] for e in E}):
            grp = [e for e in E if e[2] == tt]
            for i in range(0, len(grp), group):
                part = grp[i:i + group]
                out = sfn(torch.cat([e[1] for e in part]), tt)
                (torch.cat([g[e[0]] for e in part]) * out).sum().backward()
    return float(w * r1 * r2), float(r1), float(r2), float(A)


def plan_n(x, t, u, vs, p, k=1):
    """same as plan() but for any number of probes vs (list of tensors)"""
    hu, hv = 256.0 * p, p
    E = [("tp", x, t + k), ("tm", x, t - k), ("s0", x, t), ("up", x + hu * u, t), ("um", x - hu * u, t)]
    for j, v in enumerate(vs):
        E += [(f"pp{j}", x + hu * u + hv * v, t), (f"pm{j}", x + hu * u - hv * v, t),
              (f"mp{j}", x - hu * u + hv * v, t), (f"mm{j}", x - hu * u - hv * v, t)]
    return E


def residuals_n(s, x, u, vs, p, k, beta):
    """r(u, v_j) for every probe j (same formula as residuals()); returns (list of r_j, lhs, terms dict)"""
    hu, hv = 256.0 * p, p
    A = (u * (s["tp"] - s["tm"])).sum() / (2 * k * DTAU)
    Ju = (s["up"] - s["um"]) / (2 * hu)
    us, Jt = (u * s["s0"]).sum(), (Ju * (x + 2 * s["s0"])).sum()
    rs, d2s = [], []
    for j, v in enumerate(vs):
        D2 = (v * (s[f"pp{j}"] - s[f"pm{j}"] - s[f"mp{j}"] + s[f"mm{j}"])).sum() / (4 * hu * hv)
        rs.append(A - 0.5 * beta * (us + Jt + D2)); d2s.append(D2)
    return rs, A, {"us": 0.5 * beta * us, "Jt": 0.5 * beta * Jt, "D2": [0.5 * beta * d for d in d2s]}


def make_score_fn(unet, ac, ctx_value, amp=False, chunk=4):
    """score function s(x, t) of the network for one class label (context value); fp32 by default (A4: fp16 unusable).
    Batches larger than `chunk` are split (non-flash attention at 64x64 needs ~0.5 GB per image in fp32)."""
    def sfn(xb, tt):
        if xb.shape[0] > chunk:
            return torch.cat([sfn(xb[i:i + chunk], tt) for i in range(0, xb.shape[0], chunk)])
        c = torch.full((xb.shape[0], 1, 1), float(ctx_value), device=xb.device, dtype=xb.dtype)
        tb = torch.full((xb.shape[0],), int(tt), device=xb.device, dtype=torch.long)
        if amp:
            with torch.autocast("cuda", dtype=torch.float16):
                e = unet(xb, timesteps=tb, context=c).float()
        else:
            e = unet(xb, timesteps=tb, context=c)
        return -e / math.sqrt(1.0 - float(ac[tt]))
    return sfn


def sref_interp(table, t):
    """log-linear interpolation of the reference scale table {t: RMS(u.d_tau s)} used to normalise the loss"""
    ts = sorted(int(k) for k in table)
    t = min(max(t, ts[0]), ts[-1])
    for a, b in zip(ts[:-1], ts[1:]):
        if a <= t <= b:
            la, lb = math.log(table[str(a)] if str(a) in table else table[a]), math.log(table[str(b)] if str(b) in table else table[b])
            return math.exp(la + (lb - la) * (t - a) / max(b - a, 1))
    return table[str(ts[-1])] if str(ts[-1]) in table else table[ts[-1]]
''',
         "a6_classifier.py": r'''# a6_classifier.py - A6: real-data benign/malignant classifier on the CLEAN split (+ shared training helpers).
# Used as (i) the "real-only" reference of the later downstream test, (ii) the judge of class conditioning (A7/A9),
# (iii) the "mammography feature" extractor for KID-mammo.
# PRE-STATED RECIPE (no tuning, no epoch selection): torchvision ConvNeXt-Tiny with ImageNet-1k weights,
# 256 px grayscale -> 3 channels, per-image min-max, ImageNet normalisation, AdamW lr 1e-4 wd 0.05, batch 32,
# 15 epochs, cosine schedule with 1 warm-up epoch, horizontal flip + small affine, class-balanced cross-entropy,
# fp16 autocast. FINAL epoch is used. Seeds 0, 1, 2. Trained on clean TRAIN minus the patient-level VALIDATION split.
# PRE-STATED RULE (validation only - the test set is never used for a decision):
#   usable as judge if the mean VALIDATION AUC over the 3 seeds is >= 0.65.  Test AUC is reported, not used.
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, load_clean_meta, val_mask, OUT

TAG = "A6"
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


def prep(x01):
    """(B,1,H,W) any range -> per-image min-max -> 3ch ImageNet-normalised"""
    mn = x01.amin(dim=(1, 2, 3), keepdim=True); mx = x01.amax(dim=(1, 2, 3), keepdim=True)
    x = (x01 - mn) / (mx - mn + 1e-8)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD


def build():
    import torchvision
    m = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m


def auc(scores, labels):
    """Mann-Whitney AUC with tie handling (no sklearn dependency)"""
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    order = np.argsort(s); r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    for v in np.unique(s):
        m = s == v
        if m.sum() > 1:
            r[m] = r[m].mean()
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


@torch.no_grad()
def predict(m, X_u8, bs=64):
    m.eval(); out = []
    for i in range(0, len(X_u8), bs):
        xb = torch.as_tensor(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(m(prep(xb)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def augment(x, g):
    B = x.shape[0]
    u = lambda: torch.rand(B, generator=g, device=dev) * 2 - 1
    sx, sy, shx, shy = 1 + 0.05 * u(), 1 + 0.05 * u(), 0.05 * u(), 0.05 * u()
    z = torch.zeros(B, device=dev)
    th = torch.stack([torch.stack([sx, shx, z], 1), torch.stack([shy, sy, z], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False), padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=g, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)


def train_clf(X_u8, y, seed, epochs=15, bs=32, tag=TAG):
    """the fixed recipe above; returns the model after the FINAL epoch"""
    t0 = time.time()
    Xg, Yg = torch.from_numpy(X_u8).to(dev), torch.from_numpy(np.asarray(y)).long().to(dev)
    w = torch.tensor([len(y) / (2 * max(1, (np.asarray(y) == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed)
    m = build().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    spe = math.ceil(len(y) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda")
    g = torch.Generator(device=dev).manual_seed(seed)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev)
        tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = augment(Xg[idx].float().div(255).unsqueeze(1), g)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss)
        if (ep + 1) % 5 == 0 or ep == 0:
            log(tag, f"seed {seed} epoch {ep + 1}/{epochs} train loss {tl / spe:.4f} | {(time.time() - t0) / 60:.1f} min")
    del Xg, Yg
    return m


def boot_auc(p, y, n=1000, seed=0):
    r = np.random.default_rng(seed); vals = []
    i0, i1 = np.where(np.asarray(y) == 0)[0], np.where(np.asarray(y) == 1)[0]
    for _ in range(n):
        b = np.concatenate([r.choice(i0, len(i0)), r.choice(i1, len(i1))])
        vals.append(auc(p[b], np.asarray(y)[b]))
    return [float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--epochs", type=int, default=15)
    a = ap.parse_args()
    x, y, pat, _ = load_clean_meta("train"); xte, yte = load_cache("test")
    vm = val_mask(pat)
    res = {"recipe": "convnext_tiny IMAGENET1K_V1, AdamW 1e-4 wd .05, bs 32, 15 ep cosine (1 warm-up), flip+affine, "
                     "balanced CE, final epoch; trained on train minus validation patients",
           "n_train_fit": int((~vm).sum()), "n_val": int(vm.sum()), "n_val_patients": len(set(pat[vm])),
           "n_test": int(len(yte)), "seeds": {}}
    log(TAG, f"fit on {res['n_train_fit']} images | validation {res['n_val']} images ({res['n_val_patients']} patients)")
    pv_sum = np.zeros(int(vm.sum()))
    for seed in [int(s) for s in a.seeds.split(",")]:
        t0 = time.time()
        m = train_clf(x[~vm], y[~vm], seed, a.epochs)
        pv, pt = predict(m, x[vm]), predict(m, xte)
        pv_sum += pv
        res["seeds"][str(seed)] = {"val_auc": auc(pv, y[vm]), "test_auc_reported_only": auc(pt, yte),
                                   "val_balanced_acc": float(np.mean([((pv >= 0.5) == (y[vm] == 1))[y[vm] == c].mean() for c in (0, 1)])),
                                   "minutes": (time.time() - t0) / 60}
        torch.save({k: v.half() for k, v in m.state_dict().items()}, os.path.join(OUT, f"a6_clf_s{seed}.pt"))
        log(TAG, f"seed {seed}: VALIDATION AUC {res['seeds'][str(seed)]['val_auc']:.3f} | test AUC (reported only) "
                 f"{res['seeds'][str(seed)]['test_auc_reported_only']:.3f} | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
    va = [v["val_auc"] for v in res["seeds"].values()]
    res["mean_val_auc"] = float(np.mean(va)); res["sd_val_auc"] = float(np.std(va, ddof=1)) if len(va) > 1 else 0.0
    pe = pv_sum / len(va)
    res["ensemble_val_auc"] = auc(pe, y[vm]); res["ensemble_val_auc_ci95"] = boot_auc(pe, y[vm])
    res["mean_test_auc_reported_only"] = float(np.mean([v["test_auc_reported_only"] for v in res["seeds"].values()]))
    res["usable_as_judge"] = bool(res["mean_val_auc"] >= 0.65)
    save_json(res, "A6_classifier.json")
    log(TAG, f"mean VALIDATION AUC {res['mean_val_auc']:.3f} +- {res['sd_val_auc']:.3f} (ensemble {res['ensemble_val_auc']:.3f}) "
             f"-> usable as judge: {res['usable_as_judge']}")
''',
         "a6b_view.py": r'''# a6b_view.py - A6b: CC-vs-MLO view classifier on REAL images (label "MLO" = 1), same fixed recipe as A6, 1 seed,
# 8 epochs, trained on train-fit patients with a known view, checked on validation patients (never on test).
# Used by A9 to measure whether generated "malignant" images are side views (MLO) more often than real ones.
# PRE-STATED RULE: usable if validation AUC >= 0.95 (view is an easy, geometric property).
import os, sys, time
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, val_mask, OUT
from a6_classifier import train_clf, predict, auc

TAG = "A6b"
t0 = time.time()
x, y, pat, _ = load_clean_meta("train")
vl = np.load(os.path.join(OUT, "A10_view_labels.npz"))["view_train"].astype(str)
vm = val_mask(pat)
known = np.isin(vl, ["CC", "MLO"])
yv = (vl == "MLO").astype(int)
fit, val = known & ~vm, known & vm
log(TAG, f"fit {fit.sum()} images (MLO share {yv[fit].mean():.3f}) | validation {val.sum()} images")
m = train_clf(x[fit], yv[fit], seed=0, epochs=8, tag=TAG)
pv = predict(m, x[val])
A = auc(pv, yv[val])
acc = float(((pv >= 0.5) == (yv[val] == 1)).mean())
torch.save({k: v.half() for k, v in m.state_dict().items()}, os.path.join(OUT, "a6b_view.pt"))
res = {"n_fit": int(fit.sum()), "n_val": int(val.sum()), "val_auc": A, "val_accuracy": acc, "usable": bool(A >= 0.95),
       "minutes": (time.time() - t0) / 60}
save_json(res, "A6b_view.json")
log(TAG, f"view classifier: validation AUC {A:.4f} accuracy {acc:.3f} -> usable {res['usable']} | {res['minutes']:.1f} min")
''',
         "a7_generate.py": r'''# a7_generate.py - A7: generate samples from a clean baseline (DDIM 50 steps, same sampler as eval_arm.py).
# Spec "gs:n_per_class[s]", e.g. "4:200,1:200,0:50s":
#   guidance gs, n images per class; eps = eu + gs (ec - eu), null label -1 (as in training); gs = 1 -> one conditional pass.
#   suffix "s" = SAME starting noise for both labels + pixel-identity test (guidance 0 must ignore the label:
#   benign and malignant outputs must be identical). Otherwise each class has its own noise (seed + 100000*label + batch);
#   the same noise bank is used for every guidance value and every model (paired).
# Writes gen_<tag>_gs<g>.npz (x uint8, y) after EACH guidance value, a grid png, and the speed. --max_min stops early.
import os, sys, time, json, argparse
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_model, clean_ckpt, OUT

ap = argparse.ArgumentParser()
ap.add_argument("--seed_model", type=int, required=True)
ap.add_argument("--epoch", type=int, default=40)
ap.add_argument("--spec", default="4:200,1:200")
ap.add_argument("--steps", type=int, default=50)
ap.add_argument("--bs", type=int, default=16)
ap.add_argument("--noise_seed", type=int, default=2025)
ap.add_argument("--max_min", type=float, default=1e9)
ap.add_argument("--tag", default="")
a = ap.parse_args()
TAG = f"A7-gen-s{a.seed_model}{a.tag}"
dev = torch.device("cuda")
path = clean_ckpt(a.seed_model, a.epoch)
unet, ac, meta = load_model(path, dev)
unet = unet.half()
acf = ac.float(); T = acf.numel()
name = f"s{a.seed_model}_ep{a.epoch:03d}"
log(TAG, f"checkpoint {path} | epoch {meta.get('epoch')} | spec {a.spec}")


@torch.no_grad()
def ddim(label, n, gen, gs):
    x = torch.randn(n, 1, 256, 256, generator=gen, device=dev)
    ts = torch.linspace(T - 1, 0, a.steps, device=dev).round().long()
    cond = torch.full((n, 1, 1), float(label), device=dev)
    for i, t in enumerate(ts):
        ab_t = acf[t]
        ab_prev = acf[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=dev)
        if gs == 1.0:
            tb = torch.full((n,), int(t), device=dev, dtype=torch.long)
            eps = unet(x.half(), timesteps=tb, context=cond.half()).float()
        else:
            tb = torch.full((2 * n,), int(t), device=dev, dtype=torch.long)
            ctx = torch.cat([-torch.ones_like(cond), cond]).half()
            eu, ec = unet(torch.cat([x, x]).half(), timesteps=tb, context=ctx).float().chunk(2)
            eps = eu + gs * (ec - eu)
        x0 = ((x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()).clamp(-1, 1)
        eps = (x - ab_t.sqrt() * x0) / (1 - ab_t).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return ((x + 1) / 2).clamp(0, 1)


t_start = time.time()
info = {"checkpoint": path, "epoch": meta.get("epoch"), "steps": a.steps,
        "guidance_equation": "eps = eps_uncond + gs * (eps_cond - eps_uncond), null label -1", "runs": {}}
for item in a.spec.split(","):
    g, n = item.split(":"); same = n.endswith("s"); n = int(n.rstrip("s")); gs = float(g)
    if (time.time() - t_start) / 60 > a.max_min:
        log(TAG, f"time limit ({a.max_min} min) reached -> guidance {gs:g} skipped"); continue
    t0 = time.time(); imgs, labels, floats = [], [], {0: [], 1: []}
    for label in (0, 1):
        done, b = 0, 0
        while done < n:
            k = min(a.bs, n - done)
            seed = a.noise_seed + 7777 + b if same else a.noise_seed + 100000 * label + b
            out = ddim(label, k, torch.Generator(device=dev).manual_seed(seed), gs)[:, 0].cpu()
            if same:
                floats[label].append(out)
            imgs.append((out.numpy() * 255).round().astype(np.uint8)); labels += [label] * k; done += k; b += 1
    G = np.concatenate(imgs); L = np.array(labels, dtype=np.int64)
    sec = time.time() - t0
    key = f"gs{gs:g}" + ("_samenoise" if same else "")
    run = {"images": int(len(G)), "per_class": n, "same_noise": same, "minutes": sec / 60, "s_per_image": sec / len(G)}
    if same:
        d = (torch.cat(floats[0]) - torch.cat(floats[1])).abs()
        run["identity_test"] = {"max_abs_diff_float": float(d.max()), "mean_abs_diff_float": float(d.mean()),
                                "max_abs_diff_uint8": int(np.abs(G[L == 0].astype(int) - G[L == 1].astype(int)).max()),
                                "label_ignored": bool(float(d.max()) < 1e-3)}
        log(TAG, f"guidance {gs:g} same-noise identity test: max |benign - malignant| = {run['identity_test']['max_abs_diff_float']:.2e} "
                 f"-> label ignored: {run['identity_test']['label_ignored']}")
    info["runs"][key] = run
    np.savez_compressed(os.path.join(OUT, f"gen_{name}_{key}.npz"), x=G, y=L)
    from PIL import Image
    kk = min(8, n)
    Image.fromarray(np.concatenate([np.concatenate([G[j + c * n] for j in range(kk)], 1) for c in (0, 1)], 0)).save(
        os.path.join(OUT, f"grid_{name}_{key}.png"))
    log(TAG, f"guidance {gs:g}: {len(G)} images in {sec / 60:.1f} min ({sec / len(G):.2f} s/image)")
    save_json(info, f"A7_gen_{name}{a.tag}.json")
''',
         "a8_fp_clean.py": r'''# a8_fp_clean.py - A8: Fokker-Planck residual on the CLEAN baseline + the numbers Phase B needs (no training).
#  1. residual size and detectability on 6 validation images, t in {10,20,50,100,200,300}, 3 directions x 4 probes,
#     fp32 finite differences with the pre-fixed step p(t) (fp_core.p_of_t); exact-JVP spot check at t = 20 and 200
#  2. S_ref(t) = RMS of u.d_tau s on a dense t grid -> the fixed normalisation of the training loss
#     L_FP = r(u,v1) r(u,v2) / S_ref(t)^2   (so every t in [10,300] gets comparable weight; ~ rho^2)
#  3. gradient diagnostics (12 training images, t stratified over [10,300]):
#     |g_FP| vs |g_base| (base = original recipe loss, mean over 4 micro-batches = one optimizer step),
#     cos(g_FP, g_base), mean pairwise cos among FP gradients (is the FP direction consistent or noise?),
#     and the lambda that makes lambda*|g_FP| equal to 10 % / 30 % / 100 % of |g_base| (candidates for the LOCK)
# PRE-STATED RULES:
#   spot check: |r_FD - r_JVP| RMS / RMS(r_JVP) <= 0.10 at both t  -> the p(t) table also holds on the clean model
#   detectability: z = mean(product)/SE > 3 at each t (reported per t)
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, load_clean_meta, val_mask, to_model_domain, minmax_pm1, load_model, clean_ckpt, OUT
import fp_core as fc

ap = argparse.ArgumentParser()
ap.add_argument("--seed_model", type=int, default=42)
ap.add_argument("--epoch", type=int, default=40)
ap.add_argument("--n_img", type=int, default=6)
ap.add_argument("--n_u", type=int, default=3)
ap.add_argument("--n_v", type=int, default=4)
ap.add_argument("--grid", default="10,20,50,100,200,300")
ap.add_argument("--sref_grid", default="10,15,20,30,50,75,100,150,200,250,300")
ap.add_argument("--n_grad", type=int, default=12)
a = ap.parse_args()
TAG = "A8"
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
dev = torch.device("cuda")
path = clean_ckpt(a.seed_model, a.epoch)
unet, ac, meta = load_model(path, dev)                     # fp32, eval mode, requires_grad False
LOGAB = torch.log(ac)
log(TAG, f"checkpoint {path} | epoch {meta.get('epoch')}")
rep = {"checkpoint": path, "epoch": meta.get("epoch"), "p_of_t": {t: fc.p_of_t(t) for t in (10, 20, 50, 100, 300)}}

xtr, ytr, _pat, _ = load_clean_meta("train")
_vm = val_mask(_pat)
xte, yte = xtr[_vm], ytr[_vm]            # residual measured on VALIDATION images (the test set stays untouched in S2)
rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.where(yte == c)[0], a.n_img // 2, replace=False) for c in (0, 1)])
X0 = to_model_domain(xte[idx], dev); LAB = [int(v) for v in yte[idx]]
gen = torch.Generator(device=dev).manual_seed(11)
H, W = X0.shape[-2:]
U = torch.randn((a.n_u, 1, H, W), generator=gen, device=dev); U = U / U.flatten(1).norm(dim=1).view(-1, 1, 1, 1)
V = (torch.randint(0, 2, (a.n_v, 1, H, W), generator=gen, device=dev) * 2 - 1).float()
NOISE = torch.randn(X0.shape, generator=gen, device=dev)
SF = {lab: fc.make_score_fn(unet, ac, lab) for lab in (0, 1)}

# ---------------- 1. residual size / detectability ----------------
rows = []
t_all = time.time()
for t in [int(v) for v in a.grid.split(",")]:
    p, beta, ab = fc.p_of_t(t), fc.beta_eff(LOGAB, t), float(ac[t])
    R, L_, terms = [], [], {"us": [], "Jt": [], "D2": []}
    for i in range(X0.shape[0]):
        x = math.sqrt(ab) * X0[i:i + 1] + math.sqrt(1 - ab) * NOISE[i:i + 1]
        for j in range(a.n_u):
            u = U[j:j + 1]
            s = fc.evaluate_nograd(SF[LAB[i]], fc.plan_n(x, t, u, [V[q:q + 1] for q in range(a.n_v)], p))
            rs, A, tm = fc.residuals_n(s, x, u, [V[q:q + 1] for q in range(a.n_v)], p, 1, beta)
            R.append([float(r) for r in rs]); L_.append(float(A))
            terms["us"].append(float(tm["us"])); terms["Jt"].append(float(tm["Jt"])); terms["D2"] += [float(d) for d in tm["D2"]]
    R = np.array(R)                                                   # (n_img*n_u, n_v)
    V_ = R.shape[1]
    P = (R.sum(1) ** 2 - (R ** 2).sum(1)) / (V_ * (V_ - 1))
    m, se = float(P.mean()), float(P.std(ddof=1) / np.sqrt(len(P)))
    lhs = float(np.sqrt(np.mean(np.square(L_))))
    rows.append({"t": t, "p": p, "z": m / (se + 1e-300), "rho": float(np.sqrt(max(m, 0))) / (lhs + 1e-300),
                 "lhs_rms": lhs, "mean_sq_residual": m, "single_probe_var_over_mean_sq": float(R.var(1, ddof=1).mean() / (m + 1e-300)),
                 "term_rms": {k: float(np.sqrt(np.mean(np.square(v)))) for k, v in terms.items()}})
    log(TAG, f"t={t:4d} p={p:g} | z={rows[-1]['z']:5.2f} rho={rows[-1]['rho']:.3f} | LHS RMS {lhs:.3e} | "
             f"{(time.time() - t_all) / 60:.1f} min")
rep["residual_rows"] = rows

# ---------------- 1b. exact-JVP spot check of the p(t) table on this model ----------------
from torch.func import jvp
def r_jvp(x, t, sfn, u, v):
    beta = fc.beta_eff(LOGAB, t)
    with torch.no_grad():
        s0 = sfn(x, t); A = (u * (sfn(x, t + 1) - sfn(x, t - 1))).sum() / (2 * fc.DTAU)
    _, Ju = jvp(lambda z: sfn(z, t), (x,), (u,))
    def g(z):
        _, jv = jvp(lambda w: sfn(w, t), (z,), (v,))
        return (v * jv).sum()
    _, d2 = jvp(g, (x,), (u,))
    return float(A - 0.5 * beta * ((u * s0).sum() + (Ju * (x + 2 * s0)).sum() + d2))
spot = []
try:
    for t in (20, 200):
        ab = float(ac[t]); e_fd, e_ref = [], []
        for i in (0, X0.shape[0] - 1):
            x = math.sqrt(ab) * X0[i:i + 1] + math.sqrt(1 - ab) * NOISE[i:i + 1]
            for j in range(2):
                u, v = U[j:j + 1], V[j:j + 1]
                s = fc.evaluate_nograd(SF[LAB[i]], fc.plan_n(x, t, u, [v], fc.p_of_t(t)))
                e_fd.append(float(fc.residuals_n(s, x, u, [v], fc.p_of_t(t), 1, fc.beta_eff(LOGAB, t))[0][0]))
                e_ref.append(r_jvp(x, t, SF[LAB[i]], u, v))
        e_fd, e_ref = np.array(e_fd), np.array(e_ref)
        rel = float(np.sqrt(np.mean((e_fd - e_ref) ** 2)) / (np.sqrt(np.mean(e_ref ** 2)) + 1e-300))
        spot.append({"t": t, "rel_err_fd_vs_jvp": rel, "pass": bool(rel <= 0.10)})
        log(TAG, f"JVP spot check t={t}: rel err {rel:.4f} -> {'PASS' if rel <= 0.10 else 'FAIL'}")
except Exception as e:
    spot.append({"error": repr(e)[:300]})
rep["jvp_spot_check"] = spot

# ---------------- 2. S_ref(t) table (fixed loss normalisation) ----------------
sref = {}
with torch.no_grad():
    for t in [int(v) for v in a.sref_grid.split(",")]:
        ab = float(ac[t]); vals = []
        for i in range(X0.shape[0]):
            x = math.sqrt(ab) * X0[i:i + 1] + math.sqrt(1 - ab) * NOISE[i:i + 1]
            d = (SF[LAB[i]](x, t + 1) - SF[LAB[i]](x, t - 1)) / (2 * fc.DTAU)
            vals += [float((U[j:j + 1] * d).sum()) for j in range(a.n_u)]
        sref[str(t)] = float(np.sqrt(np.mean(np.square(vals))))
rep["S_ref_table"] = sref
log(TAG, "S_ref(t): " + ", ".join(f"{k}: {v:.3g}" for k, v in sref.items()))

# ---------------- 3. gradient diagnostics ----------------
from torchmetrics.functional.image import multiscale_structural_similarity_index_measure as msssim
_sob = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
KX, KY = (_sob[None, None] / 8).to(dev), (_sob.t()[None, None] / 8).to(dev)
def hf_loss(x, y):
    gx = F.conv2d(x, KX, padding=1) - F.conv2d(y, KX, padding=1)
    gy = F.conv2d(x, KY, padding=1) - F.conv2d(y, KY, padding=1)
    return (gx.pow(2) + gy.pow(2)).mean()
for q in unet.parameters():
    q.requires_grad_(True)
unet.train()
params = [q for q in unet.parameters()]
acf = ac.float()
def flat_grad():
    g = torch.cat([q.grad.detach().flatten() if q.grad is not None else torch.zeros(q.numel(), device=dev)
                   for q in params]).bfloat16().cpu()    # bf16: fp32 range (fp16 would underflow)
    unet.zero_grad(set_to_none=True)
    return g
Xg, Yg = torch.from_numpy(xtr).to(dev), torch.from_numpy(ytr).long().to(dev)
gg = torch.Generator(device=dev).manual_seed(5)
SCALE = 1024.0
def base_grad():
    """gradient of the ORIGINAL recipe loss on one micro-batch of 2 (as in train_arm2.py, without augmentation)"""
    idx = torch.randint(0, Xg.shape[0], (2,), generator=gg, device=dev)
    x0 = minmax_pm1(Xg[idx].float().unsqueeze(1)); y = Yg[idx]
    t = torch.randint(0, 1000, (2,), generator=gg, device=dev)
    eps = torch.randn(x0.shape, generator=gg, device=dev)
    drop = torch.rand(2, generator=gg, device=dev) < 0.15
    cond = torch.where(drop, torch.full_like(y, -1), y).float().view(2, 1, 1)
    ab = acf[t].view(-1, 1, 1, 1); xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
    with torch.autocast("cuda", dtype=torch.float16):
        eh = unet(xt, timesteps=t, context=cond)
    eh = eh.float()
    x0h = ((xt - (1 - ab).sqrt() * eh) / ab.sqrt()).clamp(-1, 1)
    loss = 0.7 * F.mse_loss(eh, eps) + 0.2 * (1 - msssim((x0h + 1) / 2, (x0 + 1) / 2, data_range=2.0, gaussian_kernel=False,
                                                      normalize="relu")) + 0.05 * hf_loss(x0h, x0)
    (loss * SCALE).backward()
    for q in params:
        if q.grad is not None:
            q.grad.div_(SCALE)
    return flat_grad(), float(loss)

bins = [(10, 20), (21, 50), (51, 100), (101, 150), (151, 225), (226, 300)]
rg = np.random.default_rng(1)
G_fp, G_base, info = [], [], []
t0 = time.time()
for n in range(a.n_grad):
    lo, hi = bins[n % len(bins)]
    t = int(rg.integers(lo, hi + 1)); i = int(rg.integers(0, len(ytr)))
    ab = float(ac[t])
    x0 = minmax_pm1(torch.from_numpy(xtr[i:i + 1]).float().unsqueeze(1).to(dev))
    x = math.sqrt(ab) * x0 + math.sqrt(1 - ab) * torch.randn(x0.shape, generator=gg, device=dev)
    u = torch.randn(x.shape, generator=gg, device=dev); u = u / u.norm()
    v1 = (torch.randint(0, 2, x.shape, generator=gg, device=dev) * 2 - 1).float()
    v2 = (torch.randint(0, 2, x.shape, generator=gg, device=dev) * 2 - 1).float()
    w = 1.0 / fc.sref_interp(sref, t) ** 2
    Lfp, r1, r2, A = fc.fp_loss_and_grad(SF[int(ytr[i])], x, t, u, v1, v2, fc.p_of_t(t), 1, fc.beta_eff(LOGAB, t), w=w)
    G_fp.append(flat_grad())
    gb, lb = base_grad()
    G_base.append(gb)
    info.append({"t": t, "label": int(ytr[i]), "L_fp_normalised": Lfp, "r1": r1, "r2": r2, "lhs": A, "base_loss": lb})
    log(TAG, f"grad sample {n + 1}/{a.n_grad}: t={t} L_fp={Lfp:+.4f} | {(time.time() - t0) / 60:.1f} min")

# Gram matrix of all gradient vectors, computed in parameter chunks on the GPU
Gall = G_fp + G_base; K = len(Gall); D = Gall[0].numel()
gram = torch.zeros(K, K, dtype=torch.float64, device=dev)
step = 20_000_000
for s0 in range(0, D, step):
    C = torch.stack([g[s0:s0 + step] for g in Gall]).to(dev).float()
    gram += (C @ C.T).double()
gram = gram.cpu().numpy()
nf = a.n_grad
norm = np.sqrt(np.diag(gram))
cos = gram / np.outer(norm, norm)
fp_n, base_n = norm[:nf], norm[nf:]
# mean of 4 base micro-batch gradients = one optimizer step: |mean| from the Gram matrix
step_norms = []
for gidx in range(nf // 4):
    sl = list(range(nf + 4 * gidx, nf + 4 * gidx + 4))
    step_norms.append(float(np.sqrt(gram[np.ix_(sl, sl)].sum()) / 4))
off = lambda M: M[~np.eye(len(M), dtype=bool)]
gd = {"samples": info,
      "fp_grad_norm": fp_n.tolist(), "base_microbatch_grad_norm": base_n.tolist(),
      "base_optimizer_step_grad_norm(mean of 4)": step_norms,
      "cos_fp_vs_own_base": [float(cos[k, nf + k]) for k in range(nf)],
      "mean_pairwise_cos_fp": float(off(cos[:nf, :nf]).mean()),
      "mean_pairwise_cos_base": float(off(cos[nf:, nf:]).mean()),
      "cos_meanFP_vs_meanBase": float(gram[:nf, nf:].sum() / np.sqrt(gram[:nf, :nf].sum() * gram[nf:, nf:].sum()))}
ratio = float(np.median(step_norms) / np.median(fp_n))
gd["ratio_median_base_step_over_median_fp"] = ratio
gd["lambda_candidates"] = {"10%": 0.1 * ratio, "30%": 0.3 * ratio, "100%": ratio}
gd["clip_note"] = f"training clips the total gradient norm at 1.0; median optimizer-step base norm = {np.median(step_norms):.3g}"
rep["gradient_diagnostics"] = gd
log(TAG, f"median |g_fp| {np.median(fp_n):.3g} | median |g_base step| {np.median(step_norms):.3g} | ratio {ratio:.3g} | "
         f"mean cos(fp, base) {np.mean(gd['cos_fp_vs_own_base']):+.3f} | mean pairwise cos fp {gd['mean_pairwise_cos_fp']:+.3f} "
         f"(base {gd['mean_pairwise_cos_base']:+.3f})")
rep["minutes"] = (time.time() - t_all) / 60
save_json(rep, "A8_fp_clean.json")
''',
         "a9_metrics.py": r'''# a9_metrics.py - A9: baseline audit on generated images (seed-42 clean baseline, guidance 1/2/3/4). TEST SET NOT USED.
# Reference images: the clean TRAIN set, split by patient into train-fit and VALIDATION (phaseA_common.val_mask).
#  (a) CONDITIONING GATE (A6 classifiers, averaged probability):
#        AUC_gen - 0.5 >= 0.5 * (AUC_real,val - 0.5), tested with the lower bound of a 1000x bootstrap 95 % CI.
#        The 50 % is OUR chosen criterion (not an established standard). If A6 fails its own validation gate,
#        conditioning is reported descriptively only.
#  (b) VIEW: predicted MLO share (A6b view classifier) of generated vs real VALIDATION images, per class.
#  (c) INTENSITY: breast-pixel mean, saturation (share of breast pixels > 0.95), high-frequency energy, breast area,
#        border brightness - generated vs real validation, per class.
#  (d) METRIC SANITY (DINOv2 ViT-B/14, Inception-v3, mammography features = A6 seed 0 penultimate layer):
#        valid if KID(train-fit sample, validation) < KID(validation, mildest degradation) and KID rises
#        monotonically with blur (sigma .5/1/2 px) and with noise (sd .02/.05/.10).
#  (e) CALIBRATION KID of each guidance value vs the TRAIN-FIT images (not test) with equal n (100 per class),
#        plus a sample-size check for guidance 4 (100 vs 200 per class).
#  (f) MEMORISATION: nearest TRAIN image in pixel space (64x64 correlation) and in DINOv2 space (cosine distance);
#        reference = validation images -> train-fit images (different patients); share of generated images more
#        similar than the reference 99th percentile; sheet of the 8 most similar generated/train pairs.
#  (g) PRE-WRITTEN GUIDANCE RULE (recommendation for the LOCK): eligible guidance = conditioning gate passes AND
#        saturation <= 2x real AND |view-share gap| <= 0.20 in both classes; choose the eligible value with the lowest
#        calibration KID (primary feature space = DINOv2 if valid, else mammography features). If none eligible:
#        lowest calibration KID, flagged "conditioning not usable -> H3 exploratory".
# KID = unbiased MMD^2, cubic polynomial kernel, 100 subsets of 100 (same function for every feature space).
import os, sys, glob, json, math, time
import numpy as np, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, val_mask, OUT
from a6_classifier import build, prep, auc, boot_auc

TAG = "A9"
dev = torch.device("cuda")
rep = {}
t_all = time.time()
x, y, pat, _ = load_clean_meta("train")
vm = val_mask(pat)
XV, YV, XF, YF = x[vm], y[vm], x[~vm], y[~vm]
GS = {}
for f in sorted(glob.glob(os.path.join(OUT, "gen_s42_ep040_gs*.npz"))):
    k = os.path.basename(f)[len("gen_s42_ep040_"):-4]
    if "samenoise" not in k:
        GS[k] = np.load(f)
GS = {k: {"x": d["x"], "y": d["y"]} for k, d in GS.items()}
log(TAG, f"generated sets: { {k: len(v['y']) for k, v in GS.items()} } | validation {len(YV)} | train-fit {len(YF)}")
def first_n(d, n):
    keep = np.concatenate([np.where(d["y"] == c)[0][:n] for c in (0, 1)])
    return {"x": d["x"][keep], "y": d["y"][keep]}

# ---------------- classifiers ----------------
def load_clf(f):
    m = build(); m.load_state_dict({k: v.float() for k, v in torch.load(f, map_location="cpu").items()}); return m.to(dev).eval()
clfs = [load_clf(f) for f in sorted(glob.glob(os.path.join(OUT, "a6_clf_s*.pt")))]
viewm = load_clf(os.path.join(OUT, "a6b_view.pt")) if os.path.exists(os.path.join(OUT, "a6b_view.pt")) else None
@torch.no_grad()
def probs(models, X_u8, bs=64):
    out = np.zeros(len(X_u8))
    for m in models:
        ps = []
        for i in range(0, len(X_u8), bs):
            xb = torch.from_numpy(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
            with torch.autocast("cuda", dtype=torch.float16):
                ps.append(torch.softmax(m(prep(xb)).float(), 1)[:, 1].cpu().numpy())
        out += np.concatenate(ps) / len(models)
    return out

# ---------------- (a) conditioning gate ----------------
a6 = json.load(open(os.path.join(OUT, "A6_classifier.json"))) if os.path.exists(os.path.join(OUT, "A6_classifier.json")) else {}
cond = {}
if clfs:
    pv = probs(clfs, XV)
    auc_val = auc(pv, YV)
    thr = 0.5 + 0.5 * (auc_val - 0.5)
    cond["real_val"] = {"auc": auc_val, "ci95": boot_auc(pv, YV), "gate_threshold": thr,
                        "a6_usable_as_judge": a6.get("usable_as_judge")}
    for k, d in GS.items():
        pg = probs(clfs, d["x"]); ci = boot_auc(pg, d["y"])
        cond[k] = {"auc": auc(pg, d["y"]), "ci95": ci, "gate_pass": bool(ci[0] >= thr),
                   "p_mal_benign_class": float(pg[d["y"] == 0].mean()), "p_mal_malignant_class": float(pg[d["y"] == 1].mean())}
        log(TAG, f"conditioning {k}: AUC {cond[k]['auc']:.3f} CI {ci[0]:.3f}-{ci[1]:.3f} | gate (>= {thr:.3f}) "
                 f"{'PASS' if cond[k]['gate_pass'] else 'FAIL'} | real val AUC {auc_val:.3f}")
rep["conditioning"] = cond

# ---------------- (b) view ----------------
view = {}
if viewm is not None:
    pr = probs([viewm], XV)
    view["real_val"] = {nm: float((pr[YV == c] >= 0.5).mean()) for c, nm in ((0, "benign"), (1, "malignant"))}
    for k, d in GS.items():
        pg = probs([viewm], d["x"])
        view[k] = {nm: float((pg[d["y"] == c] >= 0.5).mean()) for c, nm in ((0, "benign"), (1, "malignant"))}
        view[k]["gap_benign"] = view[k]["benign"] - view["real_val"]["benign"]
        view[k]["gap_malignant"] = view[k]["malignant"] - view["real_val"]["malignant"]
        log(TAG, f"predicted MLO share {k}: benign {view[k]['benign']:.2f} (real {view['real_val']['benign']:.2f}) | malignant "
                 f"{view[k]['malignant']:.2f} (real {view['real_val']['malignant']:.2f})")
rep["view"] = view

# ---------------- (c) intensity statistics ----------------
def istats(X_u8):
    I = torch.from_numpy(X_u8).float().div(255).unsqueeze(1)
    mn = I.amin(dim=(1, 2, 3), keepdim=True); mx = I.amax(dim=(1, 2, 3), keepdim=True); I = (I - mn) / (mx - mn + 1e-8)
    mu = F.avg_pool2d(I, 7, 1, 3, count_include_pad=False); m = (mu > 0.04).float(); n = m.sum((1, 2, 3)).clamp_min(1)
    hp = I - F.avg_pool2d(I, 7, 1, 3, count_include_pad=False)
    fr = torch.ones_like(I); fr[..., 4:-4, 4:-4] = 0
    return {"mean_breast": float(((I * m).sum((1, 2, 3)) / n).mean()),
            "saturation_share": float((((I > 0.95).float() * m).sum((1, 2, 3)) / n).mean()),
            "hf_energy": float(((hp * hp * m).sum((1, 2, 3)) / n).mean()),
            "breast_area": float(m.mean((1, 2, 3)).mean()), "border_brightness": float(((I * fr).sum((1, 2, 3)) / fr.sum((1, 2, 3))).mean())}
inten = {"real_val": {nm: istats(XV[YV == c]) for c, nm in ((0, "benign"), (1, "malignant"))}}
for k, d in GS.items():
    inten[k] = {nm: istats(d["x"][d["y"] == c]) for c, nm in ((0, "benign"), (1, "malignant"))}
rep["intensity"] = inten
log(TAG, "saturation share: " + " | ".join(f"{k}: {v['benign']['saturation_share']:.3f}/{v['malignant']['saturation_share']:.3f}"
                                            for k, v in inten.items()) + "  (benign/malignant)")

# ---------------- feature extractors ----------------
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
def load_dino():
    try:
        return torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", trust_repo=True), "torch.hub dinov2_vitb14"
    except Exception as e:
        log(TAG, f"torch.hub DINOv2 failed ({repr(e)[:150]}) -> timm")
        import timm
        return timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224), "timm dinov2 vitb14"
dino, rep["dinov2_source"] = load_dino(); dino = dino.to(dev).eval()
from torchmetrics.image.kid import KernelInceptionDistance
incep = KernelInceptionDistance(feature=2048, subset_size=100).to(dev).inception
mam = clfs[0] if clfs else None
@torch.no_grad()
def feats(X01, which, bs=64):
    out = []
    for i in range(0, len(X01), bs):
        xb = X01[i:i + bs].to(dev)
        if which == "dinov2":
            z = F.interpolate(xb, size=224, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
            out.append(dino((z - MEAN) / STD).float().cpu())
        elif which == "inception":
            out.append(incep((xb.clamp(0, 1) * 255).round().to(torch.uint8).repeat(1, 3, 1, 1)).float().cpu())
        else:
            with torch.autocast("cuda", dtype=torch.float16):
                h = mam.classifier[1](mam.classifier[0](mam.avgpool(mam.features(prep(xb)))))
            out.append(h.float().cpu())
    return torch.cat(out).numpy().astype(np.float64)
def kid(fa, fb, n_sub=100, size=100, seed=0):
    r = np.random.default_rng(seed); d = fa.shape[1]; vals = []; size = min(size, len(fa), len(fb))
    for _ in range(n_sub):
        A_ = fa[r.choice(len(fa), size, replace=False)]; B_ = fb[r.choice(len(fb), size, replace=False)]
        kxx = (A_ @ A_.T / d + 1) ** 3; kyy = (B_ @ B_.T / d + 1) ** 3; kxy = (A_ @ B_.T / d + 1) ** 3
        vals.append((kxx.sum() - np.trace(kxx)) / (size * (size - 1)) + (kyy.sum() - np.trace(kyy)) / (size * (size - 1)) - 2 * kxy.mean())
    return float(np.mean(vals)), float(np.std(vals))
def gblur(X, s):
    k = int(2 * math.ceil(3 * s) + 1); ax = torch.arange(k, dtype=torch.float32) - k // 2
    g = torch.exp(-ax ** 2 / (2 * s * s)); g = g / g.sum()
    X = F.conv2d(F.pad(X, (k // 2, k // 2, 0, 0), mode="reflect"), g.view(1, 1, 1, k))
    return F.conv2d(F.pad(X, (0, 0, k // 2, k // 2), mode="reflect"), g.view(1, 1, k, 1))
to01 = lambda u8: torch.from_numpy(u8).float().div(255).unsqueeze(1)
V01 = to01(XV)
F01 = to01(XF[np.random.default_rng(0).choice(len(XF), len(XV), replace=False)])
gn = torch.Generator().manual_seed(0)
deg = {f"blur{s:g}": gblur(V01, s) for s in (0.5, 1.0, 2.0)}
deg.update({f"noise{s:g}": (V01 + s * torch.randn(V01.shape, generator=gn)).clamp(0, 1) for s in (0.02, 0.05, 0.10)})
spaces = ["dinov2", "inception"] + (["mammo"] if mam is not None else [])
sanity, calib, FEAT = {}, {}, {}
for w in spaces:
    t0 = time.time()
    fv, ff = feats(V01, w), feats(F01, w)
    ffit_all = feats(to01(XF), w)
    FEAT[w] = {"val": fv, "fit_all": ffit_all}
    row = {"floor_trainfit_vs_val": kid(ff, fv)}
    for k_, X_ in deg.items():
        row[k_] = kid(fv, feats(X_, w))
    mono = lambda ks: all(row[p][0] < row[q][0] for p, q in zip(ks[:-1], ks[1:]))
    row["valid"] = bool(row["floor_trainfit_vs_val"][0] < min(row["blur0.5"][0], row["noise0.02"][0])
                        and mono(["blur0.5", "blur1", "blur2"]) and mono(["noise0.02", "noise0.05", "noise0.1"]))
    sanity[w] = row
    for k, d in GS.items():
        fg = feats(to01(first_n(d, 100)["x"]), w)
        calib.setdefault(k, {})[w] = kid(fg, ffit_all)
        if w == "dinov2":
            FEAT[w][k] = feats(to01(d["x"]), w)
    if "gs4" in GS and (GS["gs4"]["y"] == 0).sum() >= 200:
        calib.setdefault("gs4_size_check", {})[w] = {"100_per_class": calib["gs4"][w],
                                                    "200_per_class": kid(feats(to01(first_n(GS["gs4"], 200)["x"]), w), ffit_all)}
    log(TAG, f"{w}: floor {row['floor_trainfit_vs_val'][0]:.4f} | blur {[round(row[f'blur{s:g}'][0], 4) for s in (0.5, 1, 2)]} | noise "
             f"{[round(row[f'noise{s:g}'][0], 4) for s in (0.02, 0.05, 0.1)]} -> valid {row['valid']} | calibration KID "
             + " ".join(f"{k}: {calib[k][w][0]:.4f}" for k in GS) + f" | {(time.time() - t0) / 60:.1f} min")
rep["metric_sanity"] = sanity
rep["calibration_kid_vs_trainfit"] = calib
primary = "dinov2" if sanity.get("dinov2", {}).get("valid") else ("mammo" if sanity.get("mammo", {}).get("valid") else None)
rep["primary_feature_space"] = primary or "NONE valid -> discuss before LOCK"

# ---------------- (f) memorisation ----------------
def thumbs(X_u8):
    t = X_u8.reshape(len(X_u8), 64, 4, 64, 4).mean((2, 4)).reshape(len(X_u8), -1).astype(np.float64)
    t -= t.mean(1, keepdims=True); t /= np.linalg.norm(t, axis=1, keepdims=True) + 1e-9
    return t
TA = thumbs(x)                                                    # ALL train images (the generator saw all of them)
TF = thumbs(XF)
ref_pix = (thumbs(XV) @ TF.T).max(1)                              # validation -> train-fit (different patients)
q99_pix = float(np.quantile(ref_pix, 0.99))
cos = lambda A_, B_: 1 - (A_ / np.linalg.norm(A_, axis=1, keepdims=True)) @ (B_ / np.linalg.norm(B_, axis=1, keepdims=True)).T
ref_dino = cos(FEAT["dinov2"]["val"], FEAT["dinov2"]["fit_all"]).min(1)
q01_dino = float(np.quantile(ref_dino, 0.01))
fdino_all = np.concatenate([FEAT["dinov2"]["fit_all"], FEAT["dinov2"]["val"]])
xall_order = np.concatenate([XF, XV])
mem = {"reference_pixel_corr_q99": q99_pix, "reference_dino_dist_q01": q01_dino}
best = []
for k, d in GS.items():
    cg = thumbs(d["x"]) @ thumbs(xall_order).T
    pc, pi = cg.max(1), cg.argmax(1)
    dd = cos(FEAT["dinov2"][k], fdino_all).min(1)
    mem[k] = {"share_pixel_above_ref_q99": float((pc > q99_pix).mean()), "max_pixel_corr": float(pc.max()),
              "share_dino_below_ref_q01": float((dd < q01_dino).mean()), "min_dino_dist": float(dd.min())}
    best += [(float(pc[i]), k, i, int(pi[i])) for i in np.argsort(-pc)[:8]]
best = sorted(best, reverse=True)[:8]
from PIL import Image
sheet = np.concatenate([np.concatenate([GS[k]["x"][i], xall_order[j]], 1) for _, k, i, j in best], 0)
Image.fromarray(sheet).save(os.path.join(OUT, "memorisation_top8_generated_vs_nearest_train.png"))
mem["top8"] = [{"set": k, "corr": c} for c, k, _, _ in best]
rep["memorisation"] = mem
log(TAG, "memorisation (pixel): " + " ".join(f"{k}: {mem[k]['share_pixel_above_ref_q99']:.3f} (max r {mem[k]['max_pixel_corr']:.3f})" for k in GS)
         + f" | reference q99 r {q99_pix:.3f}")

# ---------------- (g) guidance rule ----------------
elig, why = [], {}
for k in GS:
    ok_c = cond.get(k, {}).get("gate_pass", False)
    rs = max(inten["real_val"]["benign"]["saturation_share"], inten["real_val"]["malignant"]["saturation_share"])
    ok_s = max(inten[k]["benign"]["saturation_share"], inten[k]["malignant"]["saturation_share"]) <= 2 * rs + 1e-6
    ok_v = bool(view.get(k)) and abs(view[k]["gap_benign"]) <= 0.20 and abs(view[k]["gap_malignant"]) <= 0.20
    why[k] = {"conditioning_gate": ok_c, "saturation_ok": bool(ok_s), "view_gap_ok": bool(ok_v)}
    if ok_c and ok_s and ok_v:
        elig.append(k)
sp = primary or "inception"
pick = min(elig or list(GS), key=lambda k: calib[k][sp][0])
rep["guidance_rule"] = {"criteria_per_guidance": why, "eligible": elig, "feature_space": sp, "recommended": pick,
                        "flag": None if elig else "no guidance value eligible -> conditioning not usable -> H3 exploratory"}
log(TAG, f"guidance rule: eligible {elig} -> recommended {pick}" + ("" if elig else " (FLAG: none eligible)"))
rep["minutes"] = (time.time() - t_all) / 60
save_json(rep, "A9_metrics.json")
''',
         "a10_metadata.py": r'''# a10_metadata.py - A10 (CPU): what differs between benign and malignant images in the REAL clean data?
# For every clean image (same order as the cache): view (CC/MLO), laterality, breast density (1-4), abnormality type
# (calc/mass) from the CBIS CSVs, and digitiser (dicom_info SecondaryCaptureDeviceManufacturer). Reported by class for
# train-fit / validation / test, with a patient-cluster bootstrap 95 % CI for the malignant-minus-benign MLO share.
# Also writes the per-image view labels used to train the view classifier (A6b).
import os, sys, ast, json, subprocess, collections
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, val_mask, OUT

TAG = "A10"
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True, capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached: Add Input -> CBIS-DDSM Breast Cancer Image Dataset (needed for the metadata)")
RAW = os.path.dirname(os.path.dirname(hit[0])); JPEG = os.path.join(RAW, "jpeg")
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
full = info[info["SeriesDescription"] == "full mammogram images"]
by_name = full.set_index("PatientName")["image_path"].to_dict()
by_uid = full.set_index("SeriesInstanceUID")["image_path"].to_dict()
scan_col = "SecondaryCaptureDeviceManufacturer" if "SecondaryCaptureDeviceManufacturer" in info.columns else None
scanner_of = info.set_index("image_path")[scan_col].astype(str).to_dict() if scan_col else {}
jpeg_dirs = set(os.listdir(JPEG))
jp = lambda rel: os.path.join(JPEG, str(rel).replace("CBIS-DDSM/jpeg/", ""))

def col(df, *names):
    low = {c.lower().strip(): c for c in df.columns}
    for n in names:
        if n in low:
            return low[n]
    return None

attr = collections.defaultdict(lambda: collections.defaultdict(set))     # rel -> field -> values
for c in ["calc_case_description_train_set", "mass_case_description_train_set", "calc_case_description_test_set",
          "mass_case_description_test_set"]:
    df = pd.read_csv(os.path.join(RAW, "csv", c + ".csv"))
    cv, cs, cd, ct = col(df, "image view"), col(df, "left or right breast"), col(df, "breast_density", "breast density"), \
        col(df, "abnormality type")
    for _, r in df.iterrows():
        parts = str(r["image file path"]).strip().split("/")
        rel = by_name.get(parts[0])
        if rel is not None and not os.path.exists(jp(rel)):
            rel = None
        if rel is None:
            for comp in parts[1:3]:
                if comp in by_uid and os.path.exists(jp(by_uid[comp])):
                    rel = by_uid[comp]; break
        if rel is None:
            for comp in parts[1:3]:
                if comp in jpeg_dirs:
                    fs = [f for f in os.listdir(os.path.join(JPEG, comp)) if f.lower().endswith(".jpg")]
                    if len(fs) == 1:
                        rel = os.path.join(comp, fs[0]); break
        if rel is None:
            continue
        a = attr[rel]
        a["view"].add(str(r[cv]).strip().upper()); a["side"].add(str(r[cs]).strip().upper())
        if cd: a["density"].add(str(r[cd]).strip())
        if ct: a["abn"].add(str(r[ct]).strip().lower())
        sc = scanner_of.get(rel) or scanner_of.get("CBIS-DDSM/jpeg/" + rel)
        a["scanner"].add(str(sc) if sc not in (None, "nan") else "unknown")

def one(vals):
    vals = {v for v in vals if v not in ("", "NAN", "nan")}
    return next(iter(vals)) if len(vals) == 1 else ("mixed" if vals else "unknown")

rep, labels = {}, {}
for split in ("train", "test"):
    man = pd.read_csv(os.path.join("/kaggle/working/clean_data", f"manifest_{split}.csv"))
    _, y, pat, md5 = load_clean_meta(split)
    assert (man.md5.astype(str).values == md5).all(), "manifest order != cache order"
    recs = []
    for files in man.files:
        acc = collections.defaultdict(set)
        for rel in ast.literal_eval(files):
            for k, v in attr.get(rel, {}).items():
                acc[k] |= v
        recs.append({k: one(acc.get(k, set())) if k != "abn" else "+".join(sorted(acc.get("abn", set()))) or "unknown"
                     for k in ("view", "side", "density", "scanner", "abn")})
    M = pd.DataFrame(recs); M["label"] = y; M["patient"] = pat
    labels[split] = np.array(M["view"].tolist(), dtype=str)
    groups = {"test": np.ones(len(M), bool)} if split == "test" else {"train_fit": ~val_mask(pat), "val": val_mask(pat)}
    for gname, gm in groups.items():
        G = M[gm]; out = {}
        for lab, nm in ((0, "benign"), (1, "malignant")):
            S = G[G.label == lab]
            out[nm] = {"n": int(len(S)), "share_MLO": float((S["view"] == "MLO").mean()), "share_LEFT": float((S.side == "LEFT").mean()),
                       "density": S.density.value_counts(normalize=True).round(3).to_dict(),
                       "scanner": S.scanner.value_counts(normalize=True).round(3).to_dict(),
                       "abnormality": S.abn.value_counts(normalize=True).round(3).to_dict(),
                       "view_unknown_or_mixed": int((~S["view"].isin(["CC", "MLO"])).sum())}
        # patient-cluster bootstrap CI of (MLO share malignant - MLO share benign)
        rng = np.random.default_rng(0)
        P = G.assign(mlo=(G["view"] == "MLO").astype(int)).groupby("patient").apply(
            lambda d: pd.Series({"nb": (d.label == 0).sum(), "nbm": ((d.label == 0) & (d.mlo == 1)).sum(),
                                 "nm": (d.label == 1).sum(), "nmm": ((d.label == 1) & (d.mlo == 1)).sum()})).to_numpy(float)
        diffs = []
        for _ in range(2000):
            S_ = P[rng.integers(0, len(P), len(P))].sum(0)
            if S_[0] > 0 and S_[2] > 0:
                diffs.append(S_[3] / S_[2] - S_[1] / S_[0])
        out["MLO_share_malignant_minus_benign"] = float(out["malignant"]["share_MLO"] - out["benign"]["share_MLO"])
        out["MLO_diff_ci95_patient_bootstrap"] = ([float(np.quantile(diffs, 0.025)), float(np.quantile(diffs, 0.975))]
                                                  if diffs else [float("nan"), float("nan")])
        rep[gname] = out
        log(TAG, f"{gname}: MLO share benign {out['benign']['share_MLO']:.3f} (n={out['benign']['n']}) | malignant "
                 f"{out['malignant']['share_MLO']:.3f} (n={out['malignant']['n']}) | diff {out['MLO_share_malignant_minus_benign']:+.3f} "
                 f"CI {out['MLO_diff_ci95_patient_bootstrap'][0]:+.3f}..{out['MLO_diff_ci95_patient_bootstrap'][1]:+.3f}")
        log(TAG, f"{gname}: density benign {out['benign']['density']} | malignant {out['malignant']['density']}")
        log(TAG, f"{gname}: scanner benign {out['benign']['scanner']} | malignant {out['malignant']['scanner']}")
np.savez(os.path.join(OUT, "A10_view_labels.npz"), view_train=labels["train"], view_test=labels["test"])
save_json(rep, "A10_metadata.json")
''',
         "report2.py": r'''# report2.py - S2 baseline-audit report (applies only the pre-stated rules written in each script header)
import os, sys, json
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import OUT

L = []
P = lambda *m: L.append(" ".join(str(x) for x in m))
def J(n):
    try:
        return json.load(open(os.path.join(OUT, n)))
    except Exception:
        return None
P("=" * 100); P("S2 BASELINE AUDIT REPORT (clean baseline seed 42; the TEST SET is not used for any decision)"); P("=" * 100)

r = J("A10_metadata.json")
P("\n[A10] REAL DATA BY CLASS (view / laterality / density / digitiser)")
if r is None:
    P("  missing (see logs/A10.log)")
else:
    for g, v in r.items():
        P(f"  {g}: MLO share benign {v['benign']['share_MLO']:.3f} (n={v['benign']['n']}) | malignant {v['malignant']['share_MLO']:.3f} "
          f"(n={v['malignant']['n']}) | malignant-benign {v['MLO_share_malignant_minus_benign']:+.3f} "
          f"CI {v['MLO_diff_ci95_patient_bootstrap'][0]:+.3f}..{v['MLO_diff_ci95_patient_bootstrap'][1]:+.3f}")
        for f in ("density", "scanner", "abnormality"):
            P(f"     {f}: benign {v['benign'][f]} | malignant {v['malignant'][f]}")
        P(f"     LEFT share: benign {v['benign']['share_LEFT']:.3f} | malignant {v['malignant']['share_LEFT']:.3f}")

r = J("A6_classifier.json")
P("\n[A6] PATHOLOGY CLASSIFIER (rule: mean VALIDATION AUC >= 0.65)")
if r is None:
    P("  missing (see logs/A6.log)")
else:
    for s, v in r["seeds"].items():
        P(f"  seed {s}: validation AUC {v['val_auc']:.3f} | test AUC (reported only) {v['test_auc_reported_only']:.3f} | {v['minutes']:.1f} min")
    P(f"  mean validation AUC {r['mean_val_auc']:.3f} +- {r['sd_val_auc']:.3f} | ensemble {r['ensemble_val_auc']:.3f} "
      f"CI {r['ensemble_val_auc_ci95'][0]:.3f}-{r['ensemble_val_auc_ci95'][1]:.3f} -> USABLE AS JUDGE: {r['usable_as_judge']}")
r = J("A6b_view.json")
P("\n[A6b] VIEW CLASSIFIER CC vs MLO (rule: validation AUC >= 0.95)")
P("  missing (see logs/A6b.log)" if r is None else f"  validation AUC {r['val_auc']:.4f} accuracy {r['val_accuracy']:.3f} -> usable {r['usable']}")

P("\n[A7] GENERATION (seed 42; speed; guidance-0 identity test)")
for f in sorted(os.listdir(OUT)):
    if f.startswith("A7_gen_") and f.endswith(".json"):
        g = J(f)
        for k, v in g["runs"].items():
            P(f"  {f[7:-5]} {k}: {v['images']} images, {v['minutes']:.1f} min, {v['s_per_image']:.2f} s/image"
              + (f" | IDENTITY TEST: max |benign-malignant| {v['identity_test']['max_abs_diff_float']:.2e} -> label ignored "
                 f"{v['identity_test']['label_ignored']}" if "identity_test" in v else ""))

r = J("A9_metrics.json")
if r is None:
    P("\n[A9] missing (see logs/A9.log)")
else:
    c = r["conditioning"]
    P("\n[A9a] CONDITIONING GATE: AUC_gen - 0.5 >= 0.5 (AUC_real,val - 0.5), lower 95 % bound")
    if "real_val" in c:
        P(f"  real validation AUC {c['real_val']['auc']:.3f} -> threshold {c['real_val']['gate_threshold']:.3f} | A6 judge usable: "
          f"{c['real_val']['a6_usable_as_judge']}")
    for k, v in c.items():
        if k != "real_val":
            P(f"  {k}: AUC {v['auc']:.3f} CI {v['ci95'][0]:.3f}-{v['ci95'][1]:.3f} -> {'PASS' if v['gate_pass'] else 'FAIL'} | mean "
              f"P(malignant): benign class {v['p_mal_benign_class']:.3f}, malignant class {v['p_mal_malignant_class']:.3f}")
    v = r["view"]
    P("\n[A9b] PREDICTED MLO SHARE (generated vs real validation)")
    if "real_val" in v:
        P(f"  real validation: benign {v['real_val']['benign']:.2f} | malignant {v['real_val']['malignant']:.2f}")
    for k, w in v.items():
        if k != "real_val":
            P(f"  {k}: benign {w['benign']:.2f} (gap {w['gap_benign']:+.2f}) | malignant {w['malignant']:.2f} (gap {w['gap_malignant']:+.2f})")
    P("\n[A9c] INTENSITY (benign / malignant)")
    for k, w in r["intensity"].items():
        P(f"  {k}: " + " | ".join(f"{s} {w['benign'][s]:.3f}/{w['malignant'][s]:.3f}" for s in w["benign"]))
    P("\n[A9d] METRIC SANITY (rule: floor < mildest degradation, monotone in blur and in noise)")
    P(f"  DINOv2 source: {r.get('dinov2_source')}")
    for w, row in r["metric_sanity"].items():
        P(f"  {w:9s}: floor {row['floor_trainfit_vs_val'][0]:.4f} | blur .5/1/2: " + " / ".join(f"{row[f'blur{s:g}'][0]:.4f}" for s in (0.5, 1, 2))
          + " | noise .02/.05/.10: " + " / ".join(f"{row[f'noise{s:g}'][0]:.4f}" for s in (0.02, 0.05, 0.1)) + f" -> VALID {row['valid']}")
    P(f"  PRIMARY FEATURE SPACE: {r['primary_feature_space']}")
    P("\n[A9e] CALIBRATION KID vs TRAIN-FIT images (100 per class; NOT the test set)")
    for k, row in r["calibration_kid_vs_trainfit"].items():
        if k == "gs4_size_check":
            for w, z in row.items():
                P(f"  size check gs4 {w}: 100/class {z['100_per_class'][0]:.4f} vs 200/class {z['200_per_class'][0]:.4f}")
        else:
            P(f"  {k}: " + " | ".join(f"{w} {z[0]:.4f}+-{z[1]:.4f}" for w, z in row.items()))
    m = r["memorisation"]
    P(f"\n[A9f] MEMORISATION: reference (validation->train-fit) pixel corr q99 {m['reference_pixel_corr_q99']:.3f}, "
      f"DINOv2 dist q01 {m['reference_dino_dist_q01']:.4f}")
    for k, z in m.items():
        if isinstance(z, dict):
            P(f"  {k}: share above pixel q99 {z['share_pixel_above_ref_q99']:.3f} (max r {z['max_pixel_corr']:.3f}) | share below "
              f"DINOv2 q01 {z['share_dino_below_ref_q01']:.3f} (min {z['min_dino_dist']:.4f})")
    g = r["guidance_rule"]
    P("\n[A9g] GUIDANCE RULE (recommendation for the LOCK)")
    for k, z in g["criteria_per_guidance"].items():
        P(f"  {k}: {z}")
    P(f"  eligible {g['eligible']} | feature space {g['feature_space']} | RECOMMENDED {g['recommended']}" + (f" | FLAG: {g['flag']}" if g["flag"] else ""))

r = J("A8_fp_clean.json")
P("\n[A8] FOKKER-PLANCK ON THE CLEAN BASELINE (validation images; fp32 FD with the p(t) table)")
if r is None:
    P("  missing (see logs/A8.log)")
else:
    for c in r["jvp_spot_check"]:
        P(f"  JVP spot check: {c}")
    for w in r["residual_rows"]:
        P(f"  t={w['t']:4d} p={w['p']:.0e} | z {w['z']:5.2f} rho {w['rho']:.3f} | LHS RMS {w['lhs_rms']:.3e}")
    P("  S_ref(t): " + ", ".join(f"{k}: {v:.3g}" for k, v in r["S_ref_table"].items()))
    g = r["gradient_diagnostics"]; med = lambda z: sorted(z)[len(z) // 2]
    P(f"  gradient: median |g_FP| {med(g['fp_grad_norm']):.3g} | median |g_base| per optimizer step "
      f"{med(g['base_optimizer_step_grad_norm(mean of 4)']):.3g} | ratio {g['ratio_median_base_step_over_median_fp']:.3g}")
    P(f"  cos(g_FP, g_base) per sample: {[round(z, 3) for z in g['cos_fp_vs_own_base']]}")
    P(f"  mean pairwise cos: FP {g['mean_pairwise_cos_fp']:+.3f} | base {g['mean_pairwise_cos_base']:+.3f} | cos(mean FP, mean base) "
      f"{g['cos_meanFP_vs_meanBase']:+.3f}")
    P(f"  lambda candidates: { {k: float(f'{v:.3g}') for k, v in g['lambda_candidates'].items()} } | {g['clip_note']}")
P("\n" + "=" * 100)
P("Send this file (or S2_results.zip) and the png images to Claude. Decisions go into the LOCK document afterwards.")
txt = "\n".join(L)
open(os.path.join(OUT, "S2_report.txt"), "w").write(txt)
print(txt)
'''}
for n, s in FILES.items():
    Path(n).write_text(s)
Path("/tmp/prepare_clean.py").write_text(r'''# prepare_clean.py - CLEAN patient-level CBIS-DDSM cache (rule B, decided 19 Oct 2026 after the data audit).
#   unit            = one unique image (pixel-identical after the 256x256 resize = the same image), not one CSV row
#   resolution      = PatientName in dicom_info (as before); if missing: series UID of the CSV path -> dicom_info, or
#                     the jpeg folder named by that UID holding exactly one image (recovers the 282 skipped test rows)
#   image check     = native min side >= 1000 px and not a binary mask (a full mammogram), else rejected + reported
#   label conflicts = an image carrying both BENIGN* and MALIGNANT rows -> excluded
#   split (rule B)  = patients that occur in both official train and test CSVs go to TEST (all their images);
#                     an image that occurs in both splits therefore ends up in test; train = everything else
#   no class subsampling (classes are already near-balanced)
# Writes /tmp/train_cache.npz and /tmp/test_cache.npz (x uint8 (N,256,256), y int64) exactly like the old prepare.py,
# plus /kaggle/working/clean_data/ (compressed caches with md5 + patient, manifests, summary) so that EVERY later
# notebook can attach this output and use byte-identical data.
import os, glob, json, shutil, subprocess, hashlib, collections, time
import numpy as np, pandas as pd
from PIL import Image

OUT = "/kaggle/working/clean_data"
os.makedirs(OUT, exist_ok=True)
EXPECTED = {"train": 2336, "test": 663}          # data audit, rule B (before the full-mammogram check)

def to_tmp(src_dir):
    for s in ("train", "test"):
        d = np.load(os.path.join(src_dir, f"{s}_cache.npz"))
        np.savez(f"/tmp/{s}_cache.npz", x=d["x"], y=d["y"])
        print(f"{s}: {d['x'].shape} benign={int((d['y'] == 0).sum())} malignant={int((d['y'] == 1).sum())}")

# ---------- 0. already built in this session, or attached from an earlier session -> reuse (identical bytes) ----------
if os.path.exists(os.path.join(OUT, "clean_summary.json")):
    to_tmp(OUT); print("clean caches already built in this session"); raise SystemExit(0)
prev = sorted(glob.glob("/kaggle/input/**/clean_data/clean_summary.json", recursive=True))
if prev:
    src = os.path.dirname(prev[0])
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(OUT, f))
    to_tmp(OUT)
    print("clean caches REUSED from", src); raise SystemExit(0)

t0 = time.time()
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached: Add Input -> CBIS-DDSM Breast Cancer Image Dataset")
RAW = os.path.dirname(os.path.dirname(hit[0]))
JPEG = os.path.join(RAW, "jpeg")
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
jp = lambda rel: os.path.join(JPEG, str(rel).replace("CBIS-DDSM/jpeg/", ""))
full = info[info["SeriesDescription"] == "full mammogram images"]
by_name = full.set_index("PatientName")["image_path"].to_dict()
by_uid = full.set_index("SeriesInstanceUID")["image_path"].to_dict()
jpeg_dirs = set(os.listdir(JPEG))

# ---------- 1. every CSV row -> image file ----------
rows, how_n = [], collections.Counter()
for split, csvs in [("train", ["csv/calc_case_description_train_set.csv", "csv/mass_case_description_train_set.csv"]),
                    ("test", ["csv/calc_case_description_test_set.csv", "csv/mass_case_description_test_set.csv"])]:
    for c in csvs:
        for _, r in pd.read_csv(os.path.join(RAW, c)).iterrows():
            parts = str(r["image file path"]).strip().split("/")
            rel, how = by_name.get(parts[0]), "patientname"
            if rel is not None and not os.path.exists(jp(rel)):
                rel = None
            if rel is None:
                for comp in parts[1:3]:
                    if comp in by_uid and os.path.exists(jp(by_uid[comp])):
                        rel, how = by_uid[comp], "series_uid_dicom_info"
                        break
            if rel is None:
                for comp in parts[1:3]:
                    if comp in jpeg_dirs:
                        fs = [f for f in os.listdir(os.path.join(JPEG, comp)) if f.lower().endswith(".jpg")]
                        if len(fs) == 1:
                            rel, how = os.path.join(comp, fs[0]), "uid_folder_single_jpeg"
                            break
            if rel is None:
                how = "unresolved"
            how_n[f"{split}:{how}"] += 1
            rows.append({"split": split, "csv": os.path.basename(c), "patient": str(r["patient_id"]),
                         "label": 1 if r["pathology"] == "MALIGNANT" else 0, "rel": rel, "how": how})
R = pd.DataFrame(rows)
print("row resolution:", dict(how_n), flush=True)
Rk = R[R.rel.notna()].copy()

# ---------- 2. load every unique file once: full-mammogram check + 256x256 array + md5 ----------
arr, md5, native, reject = {}, {}, {}, {}
files = sorted(Rk.rel.unique())
for i, rel in enumerate(files):
    with Image.open(jp(rel)) as im:
        W, H = im.size
        g = im.convert("L")
        native[rel] = (W, H)
        if min(W, H) < 1000:
            reject[rel] = f"small ({W}x{H}) - not a full mammogram"
            continue
        small = np.asarray(g.resize((512, 512)))
        if np.mean((small > 30) & (small < 225)) < 0.02:
            reject[rel] = "binary image - looks like an ROI mask"
            continue
        a = np.array(g.resize((256, 256)), dtype=np.uint8)            # identical resize to the old prepare.py
    arr[rel] = a
    md5[rel] = hashlib.md5(a.tobytes()).hexdigest()
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(files)} files ({(time.time() - t0) / 60:.1f} min)", flush=True)
rej_rows = Rk[Rk.rel.isin(reject)]
Rk = Rk[~Rk.rel.isin(reject)].copy()
Rk["md5"] = Rk.rel.map(md5)

# ---------- 3. one record per image, conflicts, rule-B split ----------
ptr, pte = set(R[(R.split == "train") & R.rel.notna()].patient), set(R[(R.split == "test") & R.rel.notna()].patient)
both = ptr & pte                                    # same definition as the data audit
img = Rk.groupby("md5").agg(rel=("rel", "first"), files=("rel", lambda s: sorted(set(s))),
                            splits=("split", lambda s: sorted(set(s))), patients=("patient", lambda s: sorted(set(s))),
                            labels=("label", lambda s: sorted(set(s))), csvs=("csv", lambda s: sorted(set(s)))).reset_index()
conflict = img.labels.map(len) > 1
clean = img[~conflict].copy()
clean["label"] = clean.labels.map(lambda s: s[0])
touch = clean.patients.map(lambda ps: any(p in both for p in ps))
in_tr = clean.splits.map(lambda s: "train" in s)
in_te = clean.splits.map(lambda s: "test" in s)
clean["final_split"] = np.where(in_te | touch, "test", "train")
clean = clean.sort_values("md5").reset_index(drop=True)
assert not (set(p for ps in clean[clean.final_split == "train"].patients for p in ps) &
            set(p for ps in clean[clean.final_split == "test"].patients for p in ps)), "patient overlap remains!"

summary = {"rule": "B: patients in both official splits -> test; label-conflict images excluded; one record per "
                   "pixel-identical image; full-mammogram check", "decided": "2026-10-19",
           "row_resolution": dict(how_n), "rejected_files": len(reject), "rejected_examples": dict(list(reject.items())[:10]),
           "rejected_rows": int(len(rej_rows)), "unique_images": int(len(img)), "label_conflict_images_excluded": int(conflict.sum()),
           "patients_in_both_official_splits": len(both), "build_minutes": None}
for s in ("train", "test"):
    d = clean[clean.final_split == s]
    x = np.stack([arr[r] for r in d.rel]); y = d.label.to_numpy().astype(np.int64)
    np.savez(f"/tmp/{s}_cache.npz", x=x, y=y)
    np.savez_compressed(os.path.join(OUT, f"{s}_cache.npz"), x=x, y=y, md5=d.md5.to_numpy().astype(str),
                        patient=d.patients.map(lambda p: "|".join(p)).to_numpy().astype(str))
    m = d[["md5", "label", "patients", "files", "csvs", "splits"]].copy()
    m["native_WxH"] = d.rel.map(lambda r: "x".join(map(str, native[r])))
    m.to_csv(os.path.join(OUT, f"manifest_{s}.csv"), index=False)
    pats = set(p for ps in d.patients for p in ps)
    summary[s] = {"images": int(len(d)), "benign": int((y == 0).sum()), "malignant": int((y == 1).sum()),
                  "patients": len(pats), "expected_from_audit": EXPECTED[s], "difference": int(len(d)) - EXPECTED[s]}
    print(f"{s}: {x.shape} benign={int((y == 0).sum())} malignant={int((y == 1).sum())} patients={len(pats)} "
          f"(audit expected {EXPECTED[s]}, difference {int(len(d)) - EXPECTED[s]})", flush=True)
summary["build_minutes"] = (time.time() - t0) / 60
json.dump(summary, open(os.path.join(OUT, "clean_summary.json"), "w"), indent=1, default=str)
print(f"rejected files (full-mammogram check): {len(reject)}", dict(list(reject.items())[:5]))
print("clean caches written to /tmp and", OUT, f"({summary['build_minutes']:.1f} min)")
''')
print("wrote", ", ".join(FILES))
subprocess.run([sys.executable, "/tmp/prepare_clean.py"], check=True)      # reuses clean_data/ from the input (identical bytes)
ck = sorted(glob.glob("/kaggle/input/**/base_clean_s42_w_ep040.pt", recursive=True))
print("clean baseline seed 42:", ck[0] if ck else "!! NOT FOUND -> Add Input -> Your Work -> clean-data-baseline-retraining")
cb = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True, capture_output=True, text=True).stdout.strip()
print("CBIS-DDSM csv:", cb or "!! NOT FOUND -> Add Input -> CBIS-DDSM (needed by A10 metadata and A6b view labels)")
r = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True)
NGPU = int(r.stdout.split()[-1]) if r.returncode == 0 else 0
print("GPUs visible:", NGPU)
LOGS = "/kaggle/working/S2/logs"; os.makedirs(LOGS, exist_ok=True)
RC = {}

def run_one(name, gpu, argv, est_h):
    """one job; skipped (rc = 'skipped') if it would pass BUDGET_H (0.6 h kept for A9 + report)"""
    if used_h() + est_h > BUDGET_H - 0.6 and name != "A9":
        print(f"--- {name} SKIPPED: {used_h():.2f} h used + {est_h} h estimate would pass the budget", flush=True)
        RC[name] = "skipped"; return
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    env["CUDA_VISIBLE_DEVICES"] = "" if gpu is None else str(gpu if NGPU >= 2 else 0)
    if gpu is None:
        env["OMP_NUM_THREADS"] = "2"
    print(f">>> {name} started on {'CPU' if gpu is None else 'GPU ' + env['CUDA_VISIBLE_DEVICES']} | session {used_h():.2f} h", flush=True)
    with open(f"{LOGS}/{name}.log", "w") as fh:
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            fh.write(line); fh.flush()
            print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
        RC[name] = p.wait()
    print(f"<<< {name} return code {RC[name]} | session {used_h():.2f} h" + ("" if RC[name] == 0 else f"  !! see {LOGS}/{name}.log"), flush=True)

def run_queues(queues):
    """queues = {gpu: [(name, [script, args], est_h), ...]} -> each GPU runs its list in order, GPUs in parallel"""
    if NGPU < 2:
        queues = {0: [j for q in queues.values() for j in q]}
    ths = [threading.Thread(target=lambda g=g, q=q: [run_one(n, g, a, e) for n, a, e in q]) for g, q in queues.items()]
    for t in ths: t.start()
    for t in ths: t.join()
print("Cell 3 done.")

In [ ]:
# ===================== CELL 4 — RUN (A10 on CPU; then GPU 0 and GPU 1 queues in parallel; then A9) =====================
t0 = time.time()
run_one("A10", None, ["a10_metadata.py"], 0.1)
run_queues({
    0: [("A7-gen-a", ["a7_generate.py", "--seed_model", "42", "--spec", "4:200,1:200,0:50s", "--max_min", "130"], 1.7),
        ("A7-gen-b", ["a7_generate.py", "--seed_model", "42", "--spec", "2:100", "--tag", "_b", "--max_min", "45"], 0.5)],
    1: [("A6", ["a6_classifier.py"], 0.5),
        ("A6b", ["a6b_view.py"], 0.25),
        ("A8", ["a8_fp_clean.py", "--seed_model", "42"], 0.7),
        ("A7-gen-c", ["a7_generate.py", "--seed_model", "42", "--spec", "3:100", "--tag", "_c", "--max_min", "45"], 0.5)]})
print(f"===== queues done in {(time.time() - t0) / 3600:.2f} h | session {used_h():.2f} h =====")
if used_h() + 0.5 <= BUDGET_H:
    run_one("A9", 0, ["a9_metrics.py"], 0.5)
else:
    print("!! A9 skipped: not enough budget left"); RC["A9"] = "skipped"
print(f"===== all done | return codes {RC} | session {used_h():.2f} h =====")

In [ ]:
# ===================== CELL 5 — REPORT + zip =====================
r = subprocess.run([sys.executable, "report2.py"], cwd=WD, capture_output=True, text=True)
print(r.stdout[-40000:])
if r.returncode != 0:
    print("!! report2.py failed:", r.stderr[-1500:])
json.dump(RC, open("/kaggle/working/S2/return_codes.json", "w"), indent=1)
z = shutil.make_archive("/kaggle/working/S2_results", "zip", "/kaggle/working/S2")
print("return codes:", RC, "| ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | session {used_h():.2f} h")